<a href="https://colab.research.google.com/github/Xv-123/dl/blob/main/templates/12_linear_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/12_linear_attention.ipynb)

# 🔴 Hard: Linear Self-Attention

Implement **Linear Attention** — O(S·D²) instead of O(S²·D), enabling efficient long-sequence processing.

Replace softmax with a **kernel feature map** $\phi$:

$$\text{LinearAttn}(Q,K,V) = \frac{\phi(Q) \left(\phi(K)^T V\right)}{\phi(Q) \cdot \sum \phi(K)}$$

### Feature map
Use $\phi(x) = \text{elu}(x) + 1$ (ensures non-negative features).

### Signature
```python
def linear_attention(Q, K, V):
    # Q: (B, S, D_k), K: (B, S, D_k), V: (B, S, D_v)
    # Returns: (B, S, D_v)
```

### Key insight
Instead of computing the $S \times S$ attention matrix, compute $\phi(K)^T V$ first (a $D_k \times D_v$ matrix), then multiply by $\phi(Q)$.

### Rules
- Must use a feature map (NOT softmax)
- Must be O(S·D²) — should run fast on long sequences
- You **may** use `F.elu`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.1 MB/s eta 0:00:00


In [2]:
import torch
import math
import torch.nn.functional as F
import torch.nn as nn


In [3]:
# # ✏️ YOUR IMPLEMENTATION HERE

# def linear_attention(Q, K, V):
#     pass  # Replace this

In [9]:
def linear_feature_map(x):
    """定义 φ(x) = elu(x) + 1，保证输出全部非负"""
    return F.elu(x) + 1

In [10]:
class LinearDotProductAttention(nn.Module):
    """线性注意力，替换原DotProductAttention，无softmax，使用kernel feature map"""
    def __init__(self, dropout, **kwargs):
        super().__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, keys, values):
        # 输入形状: (B*num_heads, S, D_k) / (B*num_heads, S, D_v)
        B_head, S_q, Dk = queries.shape
        _, S_k, Dv = values.shape

        # 特征映射 φ(Q), φ(K)
        phi_Q = linear_feature_map(queries)  # (B_head, S_q, Dk)
        phi_K = linear_feature_map(keys)     # (B_head, S_k, Dk)

        # 先计算 φ(K)^T V ，避免S×S矩阵)
        phi_K_T_V = torch.bmm(phi_K.transpose(-1, -2), values)

        # 4.分子 = φ(Q) @ (φ(K)^T V)  -> (B_head, S_q, Dv)
        numerator = torch.bmm(phi_Q, phi_K_T_V)

        # 5.分母归一化项 φ(Q) · sum_{seq} φ(K)
        sum_phiK = phi_K.sum(dim=1, keepdim=True)  # (B_head, 1, Dk)
        denominator = torch.bmm(phi_Q, sum_phiK.transpose(-1, -2))  # (B_head, S_q, 1)

        # 防止除0，极小值稳定数值
        denominator = torch.clamp(denominator, min=1e-6)

        # 6.线性注意力输出 + dropout
        attn_out = numerator / denominator
        attn_out = self.dropout(attn_out)

        # 线性注意力没有显式S×S权重矩阵，占位为空
        self.attention_weights = None
        return attn_out

In [11]:
def transpose_qkv(X, num_heads):
    """为多头并行变换QKV形状"""
    X = X.reshape(X.shape[0], X.shape[1], num_heads, -1)
    X = X.permute(0, 2, 1, 3)
    return X.reshape(-1, X.shape[2], X.shape[3])


def transpose_output(X, num_heads):
    """还原多头拼接后的形状"""
    X = X.reshape(-1, num_heads, X.shape[1], X.shape[2])
    X = X.permute(0, 2, 1, 3)
    return X.reshape(X.shape[0], X.shape[1], -1)


In [14]:
class MultiHeadLinearAttention(nn.Module):
    """多头线性注意力，替换原MultiHeadAttention"""
    def __init__(self,queries_size,keys_size,value_size, num_hiddens,
                 num_heads, dropout=0, bias=False, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        # 线性注意力
        self.attention = LinearDotProductAttention(dropout)

        # # 多头QKV投影层
        # self.W_q = nn.Linear(queries_size, num_hiddens, bias=bias)
        # self.W_k = nn.Linear(keys_size, num_hiddens, bias=bias)
        # self.W_v = nn.Linear(value_size,  num_hiddens, bias=bias)
        # self.W_o = nn.Linear(num_hiddens, value_size, bias=bias)

    def forward(self, queries, keys, values):
        # QKV线性投影
        # queries = self.W_q(queries)
        # keys = self.W_k(keys)
        # values = self.W_v(values)

        # 多头形状变换
        queries = transpose_qkv(queries, self.num_heads)
        keys = transpose_qkv(keys, self.num_heads)
        values = transpose_qkv(values, self.num_heads)

        # 线性注意力前向计算
        output = self.attention(queries, keys, values)

        # 多头输出拼接 + 输出投影
        output_concat = transpose_output(output, self.num_heads)
        # return self.W_o(output_concat)
        return output_concat


In [15]:
# 🧪 Debug
Q = torch.randn(1, 8, 16)
K = torch.randn(1, 8, 16)
V = torch.randn(1, 8, 32)
num_hiddens = 16
linear_attention = MultiHeadLinearAttention(16,16,32,64,2,dropout=0)
out = linear_attention(Q,K,V)
print("Output shape:", out.shape)   # (1, 8, 32)
print("Has NaN?", torch.isnan(out).any().item())

Output shape: torch.Size([1, 8, 32])
Has NaN? False


In [16]:
from torch_judge import check
check('linear_attention')


🧪 Testing: Linear Self-Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (3.4ms)
  ✅ [2/4] No NaN or Inf (15.6ms)
  ✅ [3/4] Gradient flow (25.8ms)
  ✅ [4/4] Runs fast on long sequences (linear complexity) (32.6ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (77.5ms total)
  Progress saved. Run status() to see your dashboard.

